# 03 — Patient Retrieval Test
Tests `patient_retriever.py` on 5 sample clinical queries.
Similarity = 0.6 × ClinicalBERT cosine sim + 0.4 × ICD Jaccard overlap

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from patient_retriever import load_resources, retrieve
model, meta, index = load_resources()

In [ ]:
sample_queries = [
    ('Patient admitted with severe sepsis and multi-organ failure, requiring ICU care.', {'99591','99592','99811'}),
    ('Elderly patient with acute heart failure and fluid overload.', {'42831','40291','5849'}),
    ('Patient with COPD exacerbation and respiratory failure needing ventilator support.', {'49121','51881','9670'}),
    ('Type 2 diabetes patient with hyperglycaemic crisis and ketoacidosis.', {'25010','25001','2762'}),
    ('Post-operative patient with wound infection and fever after abdominal surgery.', {'9985','5990','78650'}),
]

for i, (qtext, qicd) in enumerate(sample_queries, 1):
    print(f'\n{"="*60}')
    print(f'QUERY {i}: {qtext}')
    print(f'{"="*60}')
    results = retrieve(qtext, qicd, model, meta, index)
    for rank, r in enumerate(results, 1):
        print(f'  Rank {rank} | Score: {r["rank_score"]:.4f} (cos={r["cos_sim"]:.4f}, icd={r["icd_jaccard"]:.4f})')
        print(f'  Subject: {r["subject_id"]} | Age: {r["age"]} | Gender: {r["gender"]} | Admission: {r["admission_type"]}')
        print(f'  ICD codes: {r["icd_codes"]}')

## Chunk Summarizer Demo
Uses `google/flan-t5-base` to compress retrieved chunks into a ~150-token summary.

In [ ]:
from dynamic_chunker import summarize_chunks

test_chunks = [
    'Patient presented with high fever, elevated WBC, and positive blood cultures indicating sepsis.',
    'Started on broad-spectrum antibiotics. Blood pressure stabilised after fluid resuscitation.',
    'ICU admission required. Ventilator support initiated on day 2 due to respiratory failure.',
]
summary = summarize_chunks(test_chunks)
print('Summary:', summary)

## Observations
- Retriever uses hybrid scoring: 0.6 × ClinicalBERT cosine similarity + 0.4 × ICD Jaccard overlap
- ICD Jaccard overlap is low (often 0.0) because our sample has only 275 patients — with more MIMIC data, overlap would increase
- Cosine similarity drives most results at this stage, which is expected
- flan-t5-base summarizer successfully compresses multi-chunk clinical notes into concise summaries
- This chunk summarizer will feed into the dual-source RAG pipeline in later phases